In [ ]:
# import pandas as pd
# import numpy as np
# import re

# data = pd.read_csv("../data/processed/cleaned_phishing_emails.csv")

# print("Dataset Shape:", data.shape)
# data.head()

Dataset Shape: (39154, 9)


,sender,receiver,date,urls,subject,body,text,clean_text,label
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",1,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...","Never agree to be a loser Buck up, your troubl...",never agree loser buck troubles caused small d...,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,Befriend Jenna Jameson \nUpgrade your sex and ...,befriend jenna jameson upgrade sex pleasures t...,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",1,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,cnncom daily top daily top cnncom top videos s...,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",1,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,Re: svn commit: r619753 - in /spamassassin/tru...,svn commit r spamassassintrunk libmailspamassa...,0
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",1,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,SpecialPricesPharmMoreinfo \nWelcomeFastShippi...,specialpricespharmmoreinfo welcomefastshipping...,1


In [2]:
# Extract sender domain
def extract_domain(email):
    email = str(email)
    
    match = re.search(r'@([\w\.-]+)', email)
    
    if match:
        return match.group(1).lower()
    
    return "unknown"

data["sender_domain"] = data["sender"].apply(extract_domain)

data[["sender", "sender_domain"]].head(10)

,sender,sender_domain
0,Young Esposito <Young@iworld.de>,iworld.de
1,Mok <ipline's1983@icable.ph>,icable.ph
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,universalnet.psi.br
3,Michael Parker <ivqrnai@pobox.com>,pobox.com
4,Gretchen Suggs <externalsep1@loanofficertool.com>,loanofficertool.com
5,Caroline Aragon <dwthaidomainnamesm@thaidomain...,thaidomainnames.com
6,Replica Watches <jhorton@thebakercompanies.com>,thebakercompanies.com
7,Daily Top 10 <acidirev_1972@tcwpg.com>,tcwpg.com
8,qydlqcws-iacfym@issues.apache.org,issues.apache.org
9,Daily Top 10 <orn|dent_1973@musicaedischi.it>,musicaedischi.it


In [3]:
# Create metadata features
data["url_count"] = pd.to_numeric(data["urls"], errors="coerce").fillna(0)

data["has_url"] = (data["url_count"] > 0).astype(int)

data["subject_length"] = data["subject"].astype(str).str.len()

data["body_length"] = data["body"].astype(str).str.len()

data["word_count"] = data["body"].astype(str).apply(
    lambda x: len(x.split())
)

data["exclamation_count"] = data["body"].astype(str).str.count("!")

data["question_count"] = data["body"].astype(str).str.count(r"\?")

data["uppercase_count"] = data["body"].astype(str).apply(
    lambda x: sum(1 for c in x if c.isupper())
)

data["character_count"] = data["body"].astype(str).str.len()

data["uppercase_ratio"] = (
    data["uppercase_count"] /
    data["character_count"].replace(0, 1)
)

data["special_char_count"] = data["body"].astype(str).apply(
    lambda x: sum(
        1 for c in x
        if not c.isalnum() and not c.isspace()
    )
)

In [4]:
# Check the features
metadata_features = [
    "url_count",
    "has_url",
    "subject_length",
    "body_length",
    "word_count",
    "exclamation_count",
    "question_count",
    "uppercase_ratio",
    "special_char_count"
]

data[metadata_features].head()

,url_count,has_url,subject_length,body_length,word_count,exclamation_count,question_count,uppercase_ratio,special_char_count
0,1,1,25,273,46,2,0,0.021978,13
1,1,1,22,82,9,0,0,0.012195,5
2,1,1,20,3918,302,0,15,0.164625,722
3,1,1,150,24418,2660,4,75,0.007535,5769
4,1,1,26,175,2,0,0,0.205714,13


In [6]:
# Compare metadata by label
data.groupby("label")[metadata_features].mean()

,url_count,has_url,subject_length,body_length,word_count,exclamation_count,question_count,uppercase_ratio,special_char_count
label,,,,,,,,,
0,0.655210,0.655210,48.086876,2542.186287,351.144120,1.088205,2.426352,0.049495,256.997401
1,0.681668,0.681668,31.616107,801.379452,83.850838,0.802399,1.935171,0.068634,110.171825


In [7]:
# TF-IDF — Text Feature Extraction
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
X_text = data["clean_text"]
y = data["label"]

In [9]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

Training samples: 31323
Testing samples: 7831


In [10]:
# Create TF-IDF
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words="english",
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_text)

X_test_tfidf = tfidf.transform(X_test_text)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (31323, 10000)
Testing TF-IDF shape: (7831, 10000)


In [11]:
# Save TF-IDF
import pickle

with open("../data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

from scipy.sparse import save_npz

save_npz(
    "../data/processed/X_train_tfidf.npz",
    X_train_tfidf
)

save_npz(
    "../data/processed/X_test_tfidf.npz",
    X_test_tfidf
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [12]:
# save metadata
train_indices = X_train_text.index
test_indices = X_test_text.index

X_train_meta = data.loc[train_indices, metadata_features]
X_test_meta = data.loc[test_indices, metadata_features]

X_train_meta.to_csv(
    "../data/processed/X_train_metadata.csv",
    index=False
)

X_test_meta.to_csv(
    "../data/processed/X_test_metadata.csv",
    index=False
)